# Post a Site-Directed Mutagenesis Design to Teselagen

So that we can use Teselagen to design primers. These builds are two-part gibsons, with one split in the antibiotic marker and the other split at the site of the mutation.

Read through the required inputs to populate this script, and then run through. Once finished, you can follow the [FolDE SDM Protocol](https://jbei.teselagen.com/client/entries/a346c918-991a-4768-bf79-42cad5662a61) in the FolDE project in Teselagen to finish turning the build into an assembly report, ordering DNA, and using the robots to finish the build.

Required inputs:
* **CAMPAIGN_ID:** Name of this campaign. Will be used in design ID and the input genbank file names, and output teselagen build names.
* **DESIGN_ID:** What you want this called in Teselagen
* **TEMPLATE_PLASMID_REPO:** A collection of genbank files for plasmids that you have in stock, that you want to use when building this next round of mutants.
    * each genbank file should be indexed at zero within the antibiotic marker - that defines the first boundary for the gibson.
    * there should be EXACTLY one CDS feature in the genbank, and it should be on the forward strand - this is the CDS that we will mutagenize. It should include the start and stop codons.
    * the genbanks should be named like <campaign_id>_<seq_id>.gb where seq_id is either WT for the wild type sequence or follows the format described above.
* **OUTPUT_CSV_FPATH:** Where to store the mapping from teselagen design name to the target seq_id. *Keep that around, it's the rosetta stone for mapping your plasmids.*
* **TESELAGEN_OTP:** A one-time-password for uploading data to teselagen. These are specific to each user and should not ever be shared. Eg, if Justin is uploading a design to Teselagen, he should get his own OTP, and remove it from the file before saving. You can get one from `Settings-> API Password` within the application.
* **TESELAGEN_PROJECT_ID:** Which project you want the design (and templates) pasted into. You can get the project ID by going to `Settings -> Projects` and modifying the table to display the `ID` column, then right-clicking and copying that ID cell value into the notebook. It should look like 631fdf1d-5b6d-4e03-8f0c-a0f23f5066ed, which is the FolDE project ID.
* **TESELAGEN_USERNAME:** Your username.
* **NEW_SEQ_IDS:** a list of seq_ids that you want to create (eg, D104G_G429R is a D->G mutation at 104 and G->R mutation at 429; allele_ids should always be sorted by locus: 104 comes before 429). This could come, for example, from FolDE in Foldy.

In [34]:
CAMPAIGN_ID = 'TY_Pop2'

DESIGN_ID = f'TY_Pop2_R2_test3'

TEMPLATE_PLASMID_REPO = 'notebooks/jacob/round1/new_plasmids'


# Wild-type amino acid sequence - this is the reference sequence for calculating mutations
WT_AMINO_ACID_SEQUENCE = '''MLDAGFVHTYIDTHLEQRQVNKIQHGFPSPRYWSRTDVPVEEIAEDRRRVRAAGRDSFVNFYVGVPYCIQTDPGKCGYCLFPVEEFQGNAALENYFGYVEREADLYREALSGATLGAVYFGGGTSNLYREPLYHRLMDLVRGLFPDIAPQADITLEGIPQLFSRAKMQAIKDSGMNRVSMGIQQVDERLNKLSGRKQTTRHVVQSLEWARELGLAANVDLIFGWPQQTVGTMLKDLQTLVSWNVYDITHYELNVGGPTDFALNRFHELPSTLANLEMYRASRDFLTDQGYEQITAYNFRKPGDPAGRGYEEGVNRFLDSMDTVGLGYAAVSFFGNSAIGTDRSWSFINHLSLPRYKQALEEGRFPVERGFSHEAADWRLAMLFRSLFGLTVNRADYRAAFGTDVYEEFATVWDGLGEYGFVEVSDEEVGLVGDGPFYTPMVQALLAEERYRALRERETRAAQARRAARRARRTGTDGDGAGTEEVVASPGTGAPADTEAPADAEAAAATPARG'''

OUTPUT_CSV_FPATH = 'notebooks/jacob/round2/TY_Pop2_R2_test.csv'

TESELAGEN_OTP = 'bcb94e25-242a-46b8-bee7-0a971795c741'
TESELAGEN_PROJECT_ID = '82d2b17e-566a-4fab-829f-5c7c951e11ce'  # FolDE project: 631fdf1d-5b6d-4e03-8f0c-a0f23f5066ed
TESELAGEN_USERNAME = 'jbr@lbl.gov'  # Replace this with your username

assert TESELAGEN_OTP, 'TESELAGEN_OTP must be set'
assert TESELAGEN_PROJECT_ID, 'TESELAGEN_PROJECT_ID must be set'

# Number of mutations to add in this round
# NUMBER_OF_MUTATIONS = 1
# NEW_SEQ_IDS = '''Q70K
# D186N
# K234E
# E310R
# E360D
# F420L
# R200K
# I182V
# D104G_G429R
# D104R_G429R
# D152E_G429R
# F265R_G429R
# F61L_F265R
# F61L_G429R
# G416R_G429R
# G429R_T458R
# G429R_T473A
# H134R_G429R
# H8K_G429R
# H8Q_G429R
# H8R_G429R
# K234E_G429R
# M320R_G429R
# M381Y_G429R
# Q184S_G429R
# Q237E_G429R
# S125P_G429R
# S371R_G429R
# T9R_G429R
# W412L_G429R
# Y128M_G429R
# Y327A_G429R
# Y405A_G429R
# Y327P_G429R'''.split('\n')


# Number of mutations to add in this round
NUMBER_OF_MUTATIONS = 2
NEW_SEQ_IDS = '''Q70K_G429R_T458R
D104G_G429R
T9R_Y128M_G429R'''.split('\n')

In [30]:
# Utility functions for mutation analysis and template mapping
import logging
from collections import defaultdict
import glob
from app.helpers.sequence_util import allele_set_to_seq_id, get_locus_from_allele_id, maybe_get_allele_id_error_message
from pathlib import Path
from app.helpers.sequence_util import sort_seq_id_list_no_verification
import pandas as pd
from Bio import SeqIO
import re


def calculate_mutations_from_genbank(wt_aa_sequence: str, genbank_path: str) -> str:
    """
    Extract the single CDS sequence from a genbank file and calculate mutations relative to WT.
    
    Parameters
    ----------
    wt_aa_sequence : str
        The wild-type amino acid sequence
    genbank_path : str
        Path to the genbank file
        
    Returns
    -------
    str
        The seq_id representing mutations (e.g., "WT", "D104G", "D104G_G429R")
        
    Raises
    ------
    ValueError
        If genbank file issues are found (no file, multiple CDS, wrong length, etc.)
    """
    if not Path(genbank_path).exists():
        raise ValueError(f"Genbank file does not exist: {genbank_path}")
    
    try:
        record = SeqIO.read(genbank_path, "genbank")
    except Exception as e:
        raise ValueError(f"Could not read genbank file {genbank_path}: {e}")
    
    # Find CDS features
    cds_features = [f for f in record.features if f.type == "CDS"]
    if len(cds_features) == 0:
        raise ValueError(f"No CDS feature found in {genbank_path}")
    if len(cds_features) > 1:
        raise ValueError(f"Multiple CDS features found in {genbank_path}, expected exactly one")
    
    cds_feature = cds_features[0]
    
    # Extract and translate CDS
    try:
        cds_nt = cds_feature.extract(record.seq)
        cds_aa = str(cds_nt.translate())
    except Exception as e:
        raise ValueError(f"Could not extract/translate CDS from {genbank_path}: {e}")
    
    # Remove potential stop codon for comparison
    if cds_aa.endswith('*'):
        cds_aa = cds_aa[:-1]
    
    wt_clean = wt_aa_sequence.strip().replace('*', '')
    
    if len(cds_aa) != len(wt_clean):
        raise ValueError(f"Length mismatch in {genbank_path}: CDS AA length {len(cds_aa)} vs WT length {len(wt_clean)}")
    
    # Find mutations using sequence_util format
    mutations = []
    for i, (wt_aa, mut_aa) in enumerate(zip(wt_clean, cds_aa)):
        if wt_aa != mut_aa:
            allele_id = f"{wt_aa}{i+1}{mut_aa}"
            # Validate using existing utility
            error = maybe_get_allele_id_error_message(wt_clean, allele_id)
            if error:
                raise ValueError(f"Invalid mutation detected in {genbank_path}: {error}")
            mutations.append(allele_id)
    
    # Use existing utility to create seq_id
    return allele_set_to_seq_id(set(mutations))


def build_genbank_template_map(template_repo_path: str, wt_aa_sequence: str) -> dict:
    """
    Build a mapping from seq_id to genbank file path for all valid genbanks in the repo.
    
    Parameters
    ----------
    template_repo_path : str
        Path to directory containing genbank files
    wt_aa_sequence : str
        The wild-type amino acid sequence
        
    Returns
    -------
    dict
        Dictionary mapping seq_id -> genbank file path
    """
    template_map = {}
    
    repo_path = Path(template_repo_path)
    if not repo_path.exists():
        logging.warning(f"Template repo path does not exist: {template_repo_path}")
        return template_map
    
    for genbank_file in repo_path.glob("*.gb"):
        try:
            seq_id = calculate_mutations_from_genbank(wt_aa_sequence, str(genbank_file))
            template_map[seq_id] = str(genbank_file)
            logging.info(f"Added template: {seq_id} -> {genbank_file.name}")
        except ValueError as e:
            logging.warning(f"Skipping {genbank_file.name}: {e}")
            continue
    
    return template_map


def count_mutations_in_seq_id(seq_id: str) -> int:
    """Count the number of mutations in a seq_id."""
    if seq_id == "WT":
        return 0
    return len(seq_id.split('_'))


def get_seq_id_to_build_tuples(seq_ids, template_map, distance):
    """
    Map each target seq_id to (base_seq_id, new_alleles) for building.
    
    Parameters
    ----------
    seq_ids : list
        List of target seq_ids to build
    template_map : dict
        Dictionary mapping seq_id -> genbank file path
    distance : int
        Number of mutations to add (distance from base template)
        
    Returns
    -------
    dict
        Dictionary mapping seq_id -> (base_seq_id, new_alleles_list)
    """
    base_seq_ids = list(template_map.keys())
    
    possible_base_frequency = defaultdict(int)
    for new_seq_id in seq_ids:
        new_seq_id_allele_set = set(new_seq_id.split('_'))
        target_mutation_count = len(new_seq_id_allele_set)
        
        # Look for bases that are exactly 'distance' mutations away
        for possible_base in base_seq_ids:
            base_mutation_count = count_mutations_in_seq_id(possible_base)
            
            # Check if this base is the right distance away
            if base_mutation_count + distance == target_mutation_count:
                possible_base_allele_set = set(possible_base.split('_')) if possible_base != "WT" else set()
                
                # Check if all base mutations are present in the target
                if possible_base_allele_set.issubset(new_seq_id_allele_set):
                    required_new_alleles = new_seq_id_allele_set - possible_base_allele_set
                    if len(required_new_alleles) == distance:
                        possible_base_frequency[possible_base] += 1

    seq_id_to_build_tuple = {}
    for new_seq_id in seq_ids:
        new_seq_id_allele_set = set(new_seq_id.split('_'))
        target_mutation_count = len(new_seq_id_allele_set)
        
        # Find the best base (most frequently usable)
        best_bases = []
        for possible_base, frequency in sorted(possible_base_frequency.items(), key=lambda x: x[1], reverse=True):
            base_mutation_count = count_mutations_in_seq_id(possible_base)
            
            if base_mutation_count + distance == target_mutation_count:
                possible_base_allele_set = set(possible_base.split('_')) if possible_base != "WT" else set()
                
                if possible_base_allele_set.issubset(new_seq_id_allele_set):
                    required_new_alleles = new_seq_id_allele_set - possible_base_allele_set
                    if len(required_new_alleles) == distance:
                        best_bases.append((possible_base, list(required_new_alleles), frequency))
        
        if best_bases:
            # Choose the most frequently usable base
            base_seq_id, new_alleles, _ = best_bases[0]
            # Sort new alleles by position using existing utility
            sorted_alleles = sorted(new_alleles, key=lambda x: get_locus_from_allele_id(x))
            seq_id_to_build_tuple[new_seq_id] = (base_seq_id, sorted_alleles)
        else:
            logging.error(f"No base sequence found for {new_seq_id} at distance {distance}. Skipping.")

    return seq_id_to_build_tuple

In [31]:
# Build the genbank template map
logging.basicConfig(level=logging.INFO)
template_map = build_genbank_template_map(TEMPLATE_PLASMID_REPO, WT_AMINO_ACID_SEQUENCE)

print(f"Built template map with {len(template_map)} templates:")
for seq_id, path in template_map.items():
    print(f"  {seq_id} -> {Path(path).name}")

# Use the new template map for building with distance parameter
seq_id_to_build_tuples = get_seq_id_to_build_tuples(NEW_SEQ_IDS, template_map, distance=NUMBER_OF_MUTATIONS)

round2_seq_dict_list = []
for seq_index, new_seq_id in enumerate(sort_seq_id_list_no_verification(NEW_SEQ_IDS)):
    assert new_seq_id in seq_id_to_build_tuples, f"Could not find build path for {new_seq_id}"
    base_seq_id, new_alleles_list = seq_id_to_build_tuples[new_seq_id]
    round2_seq_dict_list.append({
        'campaign_id': CAMPAIGN_ID,
        'design_id': DESIGN_ID,
        'seq_id': new_seq_id,
        'teselagen_plasmid_id': f'{DESIGN_ID}_{seq_index+1:04}',
        'base_seq_id': base_seq_id,
        'new_alleles': "_".join(new_alleles_list) if len(new_alleles_list) > 1 else new_alleles_list[0],
    })

round2_seq_df = pd.DataFrame(round2_seq_dict_list)

round2_seq_df.to_csv(OUTPUT_CSV_FPATH, index=False)
round2_seq_df

INFO:root:Added template: N243G -> TY_Pop2-N243G.gb
INFO:root:Added template: N190L -> TY_Pop2-N190L.gb
INFO:root:Added template: R47L -> TY_Pop2-R47L.gb
INFO:root:Added template: WT -> TY_Pop2-WT.gb
INFO:root:Added template: D46A -> TY_Pop2-D46A.gb
INFO:root:Added template: G64H -> TY_Pop2-G64H.gb
INFO:root:Added template: Q150D -> TY_Pop2-Q150D.gb
INFO:root:Added template: Y405L -> TY_Pop2-Y405L.gb
INFO:root:Added template: Q237E -> TY_Pop2-Q237E.gb
INFO:root:Added template: F61L -> TY_Pop2-F61L.gb
INFO:root:Added template: M175I -> TY_Pop2-M175I.gb
INFO:root:Added template: T410E -> TY_Pop2-T410E.gb
INFO:root:Added template: D46V -> TY_Pop2-D46V.gb
INFO:root:Added template: F58P -> TY_Pop2-F58P.gb
INFO:root:Added template: Y405R -> TY_Pop2-Y405R.gb
INFO:root:Added template: M175V -> TY_Pop2-M175V.gb
INFO:root:Added template: F364L -> TY_Pop2-F364L.gb
INFO:root:Added template: Q150G -> TY_Pop2-Q150G.gb
INFO:root:Added template: Q226G -> TY_Pop2-Q226G.gb
INFO:root:Added template: D46L

Built template map with 25 templates:
  N243G -> TY_Pop2-N243G.gb
  N190L -> TY_Pop2-N190L.gb
  R47L -> TY_Pop2-R47L.gb
  WT -> TY_Pop2-WT.gb
  D46A -> TY_Pop2-D46A.gb
  G64H -> TY_Pop2-G64H.gb
  Q150D -> TY_Pop2-Q150D.gb
  Y405L -> TY_Pop2-Y405L.gb
  Q237E -> TY_Pop2-Q237E.gb
  F61L -> TY_Pop2-F61L.gb
  M175I -> TY_Pop2-M175I.gb
  T410E -> TY_Pop2-T410E.gb
  D46V -> TY_Pop2-D46V.gb
  F58P -> TY_Pop2-F58P.gb
  Y405R -> TY_Pop2-Y405R.gb
  M175V -> TY_Pop2-M175V.gb
  F364L -> TY_Pop2-F364L.gb
  Q150G -> TY_Pop2-Q150G.gb
  Q226G -> TY_Pop2-Q226G.gb
  D46L -> TY_Pop2-D46L.gb
  N94D -> TY_Pop2-N94D.gb
  T390R -> TY_Pop2-T390R.gb
  G429R -> TY_Pop2-G429R.gb
  N94E -> TY_Pop2-N94E.gb
  S371R -> TY_Pop2-S371R.gb


,campaign_id,design_id,seq_id,teselagen_plasmid_id,base_seq_id,new_alleles
0,TY_Pop2,TY_Pop2_R2_test2,D104G_G429R,TY_Pop2_R2_test2_0001,WT,D104G_G429R
1,TY_Pop2,TY_Pop2_R2_test2,T9R_Y128M_G429R,TY_Pop2_R2_test2_0002,G429R,T9R_Y128M
2,TY_Pop2,TY_Pop2_R2_test2,Q70K_G429R_T458R,TY_Pop2_R2_test2_0003,G429R,Q70K_T458R


In [32]:
# Utility class for building Teselagen design
from pathlib import Path
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
from app.helpers.sequence_util import get_locus_from_allele_id, maybe_get_allele_id_error_message
import re
import copy
import hashlib


############################################################
# Teselagen design data classes
############################################################

class TeselagenDesignPart:
    """A part in a Teselagen construct.

    Parameters
    ----------
    nucleic_acid_seq : str, optional
        A short nucleic‑acid sequence (to be synthesised).
    gb_tuple : tuple(str, int, int), optional
        (genbank_path, start, stop) referencing a slice in an existing plasmid.
    """
    def __init__(self, nucleic_acid_seq=None, gb_tuple=None):
        if (nucleic_acid_seq is None) == (gb_tuple is None):
            raise ValueError("Provide *either* nucleic_acid_seq or gb_tuple, not both.")
        self.nucleic_acid_seq = nucleic_acid_seq
        self.gb_tuple = gb_tuple  # (path,start,stop)

    # convenient fingerprint (hashable key) ----------------------------------
    @property
    def _key(self):
        if self.nucleic_acid_seq is not None:
            return ("seq", self.nucleic_acid_seq)
        path, start, stop = self.gb_tuple
        return ("gb", Path(path).resolve().as_posix(), start, stop)

    def get_length(self) -> int:
        """Get the length of this part."""
        if self.nucleic_acid_seq:
            return len(self.nucleic_acid_seq)
        else:
            _, start, stop = self.gb_tuple
            return stop - start

    def to_dict(self):
        if self.nucleic_acid_seq:
            return {"type": "peptide", "sequence": self.nucleic_acid_seq}
        path, start, stop = self.gb_tuple
        return {"type": "reference", "path": path, "start": start, "stop": stop}

    def __repr__(self):
        if self.nucleic_acid_seq:
            return f"TeselagenDesignPart(nucleic_acid_seq={self.nucleic_acid_seq[:10]}… )"
        return f"TeselagenDesignPart(gb_tuple={self.gb_tuple})"


class TeselagenDesignConstruct:
    """Container for parts belonging to one mutant design."""

    def __init__(self, name):
        self.name = name
        self.parts = []  # list[ TeselagenDesignPart ]

    def add_part(self, part: "TeselagenDesignPart"):
        self.parts.append(part)

    def get_total_length(self) -> int:
        """Get total length of all parts in this construct."""
        return sum(part.get_length() for part in self.parts)

    def validate_structure(self, expected_mutations: int, expected_total_length: int):
        """Validate the construct structure."""
        expected_parts = 2 * expected_mutations + 1
        if len(self.parts) != expected_parts:
            raise ValueError(f"Expected {expected_parts} parts for {expected_mutations} mutations, got {len(self.parts)}")
        
        if self.get_total_length() != expected_total_length:
            raise ValueError(f"Construct length {self.get_total_length()} doesn't match expected length {expected_total_length}")

    def to_dict(self):
        return {"name": self.name, "parts": [p.to_dict() for p in self.parts]}

    def __repr__(self):
        return f"TeselagenDesignConstruct(name={self.name}, parts={self.parts})"


############################################################
# TeselagenDesignBuilder – builds mutants and can emit JSON
############################################################

class TeselagenDesignBuilder:
    """Accumulates Teselagen constructs and can mutate targets, then export."""

    def __init__(self, design_name: str):
        self.constructs: list[TeselagenDesignConstruct] = []
        self.design_name = design_name

    def build_mutant(self, starting_genbank_fpath: str, new_seq_id: str, distance: int, wt_aa_sequence: str) -> TeselagenDesignConstruct:
        """Create a mutant construct and store it internally.
        
        Parameters
        ----------
        starting_genbank_fpath : str
            Path to the starting genbank template
        new_seq_id : str  
            Target seq_id (e.g. "D104G_G429R")
        distance : int
            Number of mutations to add from the base template
        wt_aa_sequence : str
            Wild-type amino acid sequence for validation
        """
        # Get the seq_id of the starting genbank
        base_seq_id = calculate_mutations_from_genbank(wt_aa_sequence, starting_genbank_fpath)
        
        # Verify distance
        base_mutations = set(base_seq_id.split('_')) if base_seq_id != "WT" else set()
        new_mutations = set(new_seq_id.split('_'))
        
        if not base_mutations.issubset(new_mutations):
            raise ValueError(f"Base mutations {base_mutations} not subset of new mutations {new_mutations}")
        
        required_new_mutations = new_mutations - base_mutations
        if len(required_new_mutations) != distance:
            raise ValueError(f"Expected distance {distance}, but found {len(required_new_mutations)} new mutations")
        
        # Parse and sort the new mutations by position
        new_alleles = []
        for allele in required_new_mutations:
            error = maybe_get_allele_id_error_message(wt_aa_sequence, allele)
            if error:
                raise ValueError(f"Invalid allele {allele}: {error}")
            
            pos = get_locus_from_allele_id(allele)
            wt_ltr = allele[0]
            mut_ltr = allele[-1]
            new_alleles.append((pos, wt_ltr, mut_ltr, allele))
        
        # Sort by position
        new_alleles.sort(key=lambda x: x[0])
        
        # Read the starting genbank
        record = SeqIO.read(starting_genbank_fpath, "genbank")
        cds_feature = next(f for f in record.features if f.type == "CDS")
        cds_nt = cds_feature.extract(record.seq)
        cds_aa = cds_nt.translate()
        
        # Validate all mutations
        for pos, wt_ltr, mut_ltr, allele in new_alleles:
            if cds_aa[pos - 1] != wt_ltr:
                raise ValueError(f"Expected {wt_ltr} at AA pos {pos} in {starting_genbank_fpath}, found {cds_aa[pos-1]}")
        
        # Build the construct with alternating segments and mutations
        construct_name = f"{Path(starting_genbank_fpath).stem}_{new_seq_id}"
        construct = TeselagenDesignConstruct(construct_name)
        
        cds_start = int(cds_feature.location.start)
        last_end = 0
        
        for pos, wt_ltr, mut_ltr, allele in new_alleles:
            codon_start = cds_start + (pos - 1) * 3
            codon_end = codon_start + 3
            
            # Add segment before this mutation (if any)
            if last_end < codon_start:
                construct.add_part(
                    TeselagenDesignPart(gb_tuple=(starting_genbank_fpath, last_end, codon_start))
                )
            
            # Add the mutant codon
            current_codon = str(cds_nt[(pos - 1) * 3 : pos * 3])
            new_codon = self._choose_codon(current_codon, mut_ltr, cds_nt, cds_aa)
            construct.add_part(TeselagenDesignPart(nucleic_acid_seq=new_codon))
            
            last_end = codon_end
        
        # Add final segment
        construct.add_part(
            TeselagenDesignPart(gb_tuple=(starting_genbank_fpath, last_end, len(record)))
        )
        
        # Validate construct using the new method
        construct.validate_structure(distance, len(record))
        
        self.constructs.append(construct)
        return construct

    def _choose_codon(self, current_codon: str, aa_letter: str, cds_nt: Seq, cds_aa: Seq) -> str:
        aa_codon_counts = {}
        for aa_idx, aa in enumerate(cds_aa):
            if aa == aa_letter:
                codon = cds_nt[aa_idx * 3: (aa_idx + 1) * 3]
                aa_codon_counts[codon] = aa_codon_counts.get(codon, 0) + 1
        if len(aa_codon_counts) == 0:
            raise ValueError(f"No codons found for {aa_letter} in existing gene ({cds_aa})")
        
        codon_counts_sorted = sorted(aa_codon_counts.items(), key=lambda x: x[1], reverse=True)
        most_common_codon = codon_counts_sorted[0][0]
        if most_common_codon.translate() != aa_letter:
            raise ValueError(f"Bug! Most common codon for {aa_letter} in existing gene ({cds_aa}) is {most_common_codon}, which translates to {most_common_codon.translate()}")
        return str(most_common_codon)

    def to_teselagen_json(self, assembly_method="golden gate", allow_duplicates=True):
        """Produce Teselagen‑compatible JSON including *all* annotated GenBank parts."""

        part_key_to_id: dict[tuple, str] = {}
        gb_subsets: dict[str, set[tuple[int, int]]] = {}
        nucleic_parts: dict[str, str] = {}

        def _sha_id(txt: str) -> str:
            return f'p_{txt}'
        
        # aggregate parts from constructs
        for cons in self.constructs:
            for part in cons.parts:
                key = part._key
                if key in part_key_to_id:
                    continue
                if key[0] == "gb":
                    _, pth, s, e = key
                    pid = f"{Path(pth).stem}-{s}-{e}"
                    part_key_to_id[key] = pid
                    gb_subsets.setdefault(pth, set()).add((s, e))
                else:  # synthetic sequence
                    _, seq = key
                    pid = _sha_id(seq)
                    part_key_to_id[key] = pid
                    nucleic_parts[pid] = seq

        # sequences array
        sequences_json = []
        for gb_path, subset_set in gb_subsets.items():
            rec = SeqIO.read(gb_path, "genbank")
            seq_name = Path(gb_path).stem
            parts_json = []

            # add *all* annotated features as parts
            for feat in rec.features:
                if feat.type == "source":
                    continue
                start = int(feat.location.start)
                end = int(feat.location.end) - 1  # Inclusive for Teselagen
                fid = f"{seq_name}_{start}_{end}_{feat.type}"
                fname = feat.qualifiers.get("label", [feat.type])[0]
                parts_json.append({
                    "start": start,
                    "end": end,
                    "id": fid,
                    "name": fname,
                    "strand": 1 if feat.location.strand != -1 else -1,
                })

            # add subset slices actually referenced by constructs
            for (s, e) in sorted(subset_set):
                pid = part_key_to_id[("gb", Path(gb_path).resolve().as_posix(), s, e)]
                parts_json.append({
                    "start": s,
                    "end": e - 1,
                    "id": pid,
                    "name": pid,
                    "strand": 1,
                })

            sequences_json.append({
                "name": seq_name,
                "sequence": str(rec.seq),
                "parts": parts_json,
                "circular": True,
            })

        # synthetic sequences
        for pid, seq in nucleic_parts.items():
            sequences_json.append({
                "name": pid,
                "sequence": seq,
                "parts": [{"start": 0, "end": len(seq) - 1, "id": pid, "name": pid, "strand": 1}],
            })

        # Build columns (one per construct)
        columns_json = []
        for column_idx in range(max(len(construct.parts) for construct in self.constructs)):
            col_parts = []
            for construct in self.constructs:
                if column_idx < len(construct.parts):
                    part = construct.parts[column_idx]
                    pid = part_key_to_id[part._key]
                    col_parts.append({"id": pid})
                else:
                    col_parts.append({"id": ""})
            columns_json.append({
                "direction": "forward",
                "icon": "cds",
                "name": f'Part {column_idx + 1}',
                "parts": col_parts,
            })

        # Final JSON structure
        design_json = {
            "assembly_method": assembly_method,
            "columns": columns_json,
            "layout_type": "list",
            "name": self.design_name,
            "sequences": sequences_json,
        }

        return {
            "allowDuplicates": allow_duplicates,
            "designJson": design_json,
        }

In [33]:
# Populate a builder class using the template map and unified build method
builder = TeselagenDesignBuilder(DESIGN_ID)
for row_idx, row in round2_seq_df.iterrows():
    seq_id = row['seq_id']
    base_seq_id = row['base_seq_id']

    # Use the template map to get the genbank file path
    if base_seq_id in template_map:
        genbank_fpath = template_map[base_seq_id]
        # Use the unified build method for all cases
        builder.build_mutant(genbank_fpath, seq_id, NUMBER_OF_MUTATIONS, WT_AMINO_ACID_SEQUENCE)
    else:
        print(f"ERROR: No template found for base sequence {base_seq_id}")

builder.to_teselagen_json()

/Users/jacobroberts/git/foldy/.venv/lib/python3.12/site-packages/Bio/SeqFeature.py:1043: BiopythonParserWarning: Attempting to fix invalid location '14883..480' as it looks like incorrect origin wrapping. Please fix input file, this could have unintended behavior.
  warnings.warn(


{'allowDuplicates': True,
 'designJson': {'assembly_method': 'golden gate',
  'columns': [{'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 1',
    'parts': [{'id': 'TY_Pop2-WT-0-9125'},
     {'id': 'TY_Pop2-G429R-0-8840'},
     {'id': 'TY_Pop2-G429R-0-9023'}]},
   {'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 2',
    'parts': [{'id': 'p_GGC'}, {'id': 'p_CGC'}, {'id': 'p_AAG'}]},
   {'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 3',
    'parts': [{'id': 'TY_Pop2-WT-9128-10100'},
     {'id': 'TY_Pop2-G429R-8843-9197'},
     {'id': 'TY_Pop2-G429R-9026-10187'}]},
   {'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 4',
    'parts': [{'id': 'p_CGC'}, {'id': 'p_ATG'}, {'id': 'p_CGC'}]},
   {'direction': 'forward',
    'icon': 'cds',
    'name': 'Part 5',
    'parts': [{'id': 'TY_Pop2-WT-10103-15194'},
     {'id': 'TY_Pop2-G429R-9200-15194'},
     {'id': 'TY_Pop2-G429R-10190-15194'}]}],
  'layout_type': 'list',
  'name': 'TY_Pop2_R2_test

In [35]:
import requests
design = builder.to_teselagen_json()

# This script is used to post a design to the Teselagen API.
def post_design(session, design):
    """Fetch notebook entry by id."""
    url = f"{BASE_URL}/designs"
    response = session.post(url, json=design)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return []


BASE_URL = "https://jbei.teselagen.com/tg-api"

session: requests.Session = requests.Session()
session.headers.update(
    {"Content-Type": "application/json", "Accept": "application/json"}
)

# Authenticate and get the token
response: requests.Response = session.put(
    url=f"{BASE_URL}/public/auth",
    json={
        "username": TESELAGEN_USERNAME,
        "password": TESELAGEN_OTP,
        "expiresIn": "1d",
    },
)
response.raise_for_status()  # Raise an error if a problem is found
session.headers.update(
    {
        "x-tg-api-token": response.json()["token"],  # TOKEN
        "tg-project-id": TESELAGEN_PROJECT_ID
    },  
)
session.headers.pop("Content-Type", None)
del response

# post the design
post_design(session, design)


{'id': 'd6b502f6-9ccc-47e1-a89c-4ecca752e429'}